## Entorno: Data, docker y spark

Se descarga la data y se configura el entorno con docker y spark

#### Installs

Instala Jupyter (`notebook`), la dependencia necesaria para poder abrir y correr este notebook

In [ ]:
%pip install -q notebook

#### Imports

Importa las librerías estándar (subprocess, sys, pathlib) que usan las demás celdas del notebook

In [ ]:
import subprocess, sys
from pathlib import Path

#### Utilidades

Define `sh()`, la función que ejecuta comandos de shell mostrando su salida en vivo, y calcula la ruta raíz del proyecto

In [ ]:
def sh(cmd):
    p = subprocess.Popen(cmd, shell=True, cwd=proyecto, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for linea in p.stdout:
        print(linea, end="")

    p.wait()

    if p.returncode != 0:
        raise RuntimeError(p.returncode)

proyecto = Path.cwd()
if not (proyecto / "docker-compose.yml").exists() and (proyecto.parent / "docker-compose.yml").exists():
    proyecto = proyecto.parent

### 1. Descargar data

- Ejecuta `data/download.py` para descargar los archivos JSON de contrataciones

- El script salta los meses que ya están en `data/raw` y solo baja los que faltan

In [ ]:
sh(f'"{sys.executable}" data/download.py')
print(f"{len(list((proyecto / 'data' / 'raw').glob('*/*.json')))} archivos en data/raw")

### 2. Verificación docker

Comprueba que el CLI de Docker y el plugin `compose` estén instalados en la máquina

In [ ]:
sh("docker --version")
sh("docker compose version")

### 3. Iniciar cluster spark

- Levanta el cluster de Spark (master + 2 workers) y Jupyter

- Kernel picker → "Existing Jupyter Server..." → `http://localhost:8888`

In [ ]:
sh("docker compose up -d")
sh("docker compose ps")

### 4. Monitorear cluster

- Muestra el estado de los contenedores y el consumo de CPU/RAM del cluster de Spark

- UI de Spark: `http://localhost:8080` (master) y `http://localhost:4040` (job activo)

In [ ]:
sh("docker compose ps")
sh("docker stats --no-stream")

### 5. Apagar cluster

Detiene y borra los contenedores, pero la data y los resultados quedan en disco

In [ ]:
# sh("docker compose down")